Before we begin, let us execute the below cell to display information about the NVIDIA® CUDA® driver and the GPUs running on the server by running the `nvidia-smi` command. To do this, execute the cell block below by clicking on it with your mouse, and pressing Ctrl+Enter, or pressing the play button in the toolbar above. You should see some output returned below the grey cell.

In [ ]:
nvidia-smi

# Learning objectives
The **goal** of this lab is to:

- Understand what OpenMP is and how it supports parallel programming on CPUs and GPUs using directives.
- Identify where to use OpenMP directives in your programs.
- Learn the core concepts in OpenMP, including the role of threads and the fork-join model for parallel task execution.
- Implement a range of OpenMP directives to distribute workloads across threads for parallelisation.
- Differentiate between shared and private data scopes in the OpenMP memory paradigm. 
- Utilise OpenMP directives and constructs for GPU offloading and managing data between the CPU and GPU.
- Analyse the benefits of collapsing loops to improve parallelisation.

We do not intend to cover:
- Optimization techniques in details

**NOTE**: To be able to see the Nsight Systems profiler output, please download the latest version of Nsight Systems from [here](https://developer.nvidia.com/nsight-systems).

# OpenMP Directives
- OpenMP was formed in 1997 to focus on vendor-neutral Shared Memory Parallelism.
- OpenMP 4.0 in 2013 expanded its focus beyond shared memory parallel computers including accelerators. 
- The OpenMP 4.0 target construct provides the means to offload data and computation to accelerators.

OpenMP is directive based where the compiler directives appear as comments in your source code and are ignored by compilers unless you tell them otherwise - usually by specifying the appropriate compiler flag.

In this notebook we will use the OpenMP target construct to offload data and computation to GPU. Multiple compilers are in development to support OpenMP offloading to NVIDIA GPUs. We will be using NVIDIA HPC SDK compiler for this tutorial.

<summary markdown="span"><b>syntax</b></summary>
    
```!$omp directive ``` 


**!$** in Fortran are what's known as a "compiler hints" (often called pragmas, following the C/C++ convention). These are very similar to programmer comments, however, the compiler will read our pragmas. Pragmas are a way for the programmer to "guide" the compiler, without running the chance of damaging the code. If the compiler does not understand the pragma, it can ignore it, rather than throw a syntax error.

**omp** is an addition to our compiler hint, known as the “sentinel”. It specifies that this is an OpenMP pragma. Any non-OpenMP compiler will ignore this pragma. 

**directives** are commands in OpenMP that will tell the compiler to do some action. For now, we will only use directives that allow the compiler to parallelize our code.
    
For beginners new to OpenMP directives, we will introduce some terminologies and concepts before adding ```target``` directives to our code to offload onto GPU computation and data.

## OpenMP Fork-Join Model

OpenMP uses the fork-join model of parallel execution. All OpenMP programs begin as a single process: the master thread. The master thread executes sequentially until the first parallel region construct is encountered.

**FORK**: The master thread then creates a team of parallel threads. The statements in the program that are enclosed by the parallel region construct are then executed in parallel among the various team threads.

**JOIN**: When the team threads complete the statements in the parallel region construct, they synchronize and terminate, leaving only the master thread.

<img src="../../_common/images/openmp_fork_join.png" width="50%" height="50%">

## OpenMP Parallel Region

A parallel region is a block of code  executed by multiple threads. This is the fundamental OpenMP parallel construct. When a thread reaches a  ```parallel``` directive, it creates a team of threads and becomes the master of the team. The master is a member of that team. Starting from the beginning of this parallel region, the code is duplicated, and all threads will execute that code redundantly. There is an implied barrier at the end of a parallel region. Only the master thread continues execution past this point.

<summary markdown="span"><b>Syntax</b></summary>
    
```fortran
program hello
  implicit none

  integer :: nthreads

  ! Fork a team of threads
  !$omp parallel
  
    ! Obtain and print thread id
    print *, "Hello World from thread =", omp_get_thread_num()

    ! Only master thread does this
    if (omp_get_thread_num() == 0) then
       nthreads = omp_get_num_threads()
       print *, "Number of threads =", nthreads
    end if

  !$omp end parallel  ! All threads join master thread and terminate

end program hello
 ```

<img src="../../_common/images/openmp_parallel_construct.png" width="50%" height="50%">


### OpenMP Work-sharing

As described above, the ```parallel``` construct creates a team of threads, and the execution continues redundantly on all threads of the team. Ideally we would need all threads within the team to work share i.e. split the work. A work-sharing construct divides the execution of the enclosed code region among the members of the team that encounter it. Work-sharing constructs do not launch new threads but divide (“workshares”) the iterations of the  loop across the threads in the team . There is no implied barrier upon entry to a work-sharing construct, however there is an implied barrier at the end of a work-sharing construct. 

There are multiple ways to allow worksharing, the code below makes use of ```for``` to divide the iteration of a loop among threads.

<summary markdown="span"><b>Syntax</b></summary>

```fortran
! Create a team of threads
!$omp parallel
! workshare this loop across those threads.
    !$omp for
    do i=1,N
        c(i) = a(i) + b(i)
    end do
    ! end of workshare region
!$omp end parallel
! end of parallel region
```

In cases where only a single parallel worksharing region is required, the `parallel` and worksharing directive (`for` in this example) can be combined to create a single construct.

<summary markdown="span"><b>Syntax</b></summary>

```fortran
! Create a team of threads and workshare this loop across those threads.
!$omp parallel for
do i = 1, N
    c(i) = a(i) + b(i)
end do
!$omp end parallel
! end of workshare parallel region
```
    
<img src="../../_common/images/openmp_parallelfor_construct.png" width="50%" height="50%">


### OpenMP Data-sharing
In OpenMP, several constructs accept clauses that allow the user to control the data sharing. For example, you can use one of the below clauses in a *Parallel* construct.

- `private`: Declares variables to be private to each thread in a team. Private copies of the variable are initialized from the original object when entering the region.
- `shared`: Shares variables among all the threads in a team.
- `default`: Enables you to affect the data-scope attributes of variables.

<summary markdown="span"><b>Syntax</b></summary>
    
```fortran
!$omp parallel do default(shared) private(dx)
do i = 1, N 
    do j = 1, N 
        dx = a(i) + b(j)
    end do
end do
!$omp end parallel
```

## Atomic Construct
In the code, you will also require one more construct, which will help you get the right results. OpenMP atomic construct ensures that a particular variable is accessed and/or updated atomically to prevent indeterminate results and race conditions. In other words, it prevents one thread from stepping on the toes of other threads due to accessing a variable simultaneously, resulting in different results run-to-run. For example, if we want to accumulate numbers up to N, we could write the following:

<summary markdown="span"><b>Syntax</b></summary>
    
```fortran
do i = 1, N
  !$omp atomic
  cnt = cnt + 1
end do
```


# OpenMP Multicore Exercise

Let's start modifying the original code and add the OpenMP directives we introduced for multicore CPU execution.

**Click on the <b>[source code](../source_code/rdf.f90)</b> link, and start modifying the RDF code.**

Remember to pay attention to worksharing and datasharing.

Note that without changing the orginal code, you will not get the expected outcome after running the below cells. To help you modify the code, some sections are marked with `TODO: ` comments consisting of simple instructions.  Where additional modifications from the [original source code](../../_common/source_code/rdf.f90) were required they are marked with `Note: ` comments for you to review. Remember to **SAVE** your code after changes, before running the below cells.

### Compile and Run for Multicore

Having added OpenMP directives, let us compile the code. We will be using NVIDIA HPC SDK compiler for this exercise. The flags used for enabling OpenMP target offloading are as follows:

<!--
**-fopenmp** : This flag will give tell compiler to parse and act on OpenMP directive.
**-fopenmp-targets** : This flag allows us to compile our code for a specific target parallel hardware. Without this flag, the code will be compiled for multicore execution.
-->

`-mp=gpu|multicore` : Select the target device for all parallel programming paradigms used (OpenACC, OpenMP, Standard Languages)
- `gpu`             Globally set the target device to an NVIDIA GPU
- `multicore`       Globally set the target device to the host CPU

**NOTE:** `-Minfo=mp` enables OpenMP information, although it is not very verbose.

After running the cells, make sure to check the output first. You can inspect part of the compiler feedback and see what it's telling us (your compiler feedback will be similar to the below).

### Compile the code for multicore (Fortran)

In [ ]:
#Compile the code for multicore
cd ../source_code && printf "Compiling multicore version ...\n" && make clean && make rdf_f &&
printf "\nRunning the executable and validating the output\n" && ./rdf_f && cat Pair_entropy.dat

Example compiler feedback:

```
pair_gpu:
    176, !$omp parallel
```

We can see that the compiler found the compiler hint, `176, !$omp parallel`, and will attempt to parallelise the loops.

The output should be the following:

```
s2 value is -2.43191
s2bond value is -3.87015 
```

Now, let's profile the code.

In [ ]:
#profile and see output of nvptx
cd ../source_code && nsys profile -t nvtx --stats=true --force-overwrite true -o rdf_multicore_f ./rdf_f

Let's checkout the profiler's report.  Download and save the report file by holding down the Shift key and right-clicking the [report](../source_code/rdf_multicore_f.nsys-rep) link then choosing Save Link As. Once done, open it via the GUI. Have a look at the example expected profiler report below:

**Example screenshot (multicore version)**

<img src="../../_common/images/f_openmp_multicore.png">


Feel free to checkout the [solution](../source_code/SOLUTION/rdf.f90) to help you understand better.

# OpenMP Directives for GPU

Now that got some familiarity with the OpenMP programming model, let us introduce the key directives and constructs used for GPU offloading.  These additional directive are ignored by the compilers unless specified to compile for for a target device - usually by specifying the appropriate compiler flag - allowing the same code to be compiled also for multicore CPU or serial.

## Target construct

The `target` construct consists of a target directive and an execution region. `target` directive defines a target region,  a block of computation that operates within a distinct data environment and is intended to be offloaded onto a parallel computation device during execution ( GPU in our case). Data used within the region may be implicitly or explicitly mapped to the device.  OpenMP is allowed within target regions, but only a subset will run well on GPUs.

The example below shows the usage of target directive with implicitly mapped data

<summary markdown="span"><b>Syntax</b></summary>
    
```fortran
    !Moves this region of code to the GPU and implicitly maps data.
    !$omp target
        !$omp parallel do
        do i = 1, N 
            c(i) = a(i) + b(i)
        end do
    !$omp end target    
```

Note that OpenMP will **only** implicitly map data with known size at the compile-time. Any data structure that is dynamically allocated must be explicilty mapped, as explained below.

##  Target data to explicitly map the data

The `map` directive helps developers to explicitly define and reduce data copies. The `target data` construct is used to mark such regions.

<summary markdown="span"><b>Syntax</b></summary> 

```fortran
!$omp target data map(map-type: list)
```

Examples of mapping data directives are as follows: 
- `to` (list)
    - Allocates memory on the device and copies data in when entering the region, the values are not copied back
- `from` (list)
    - Allocates memory on the device and copies the data to the host when exiting the region
- `alloc` (list)
    - Allocates memory on the device. If the data is already present on the device, a reference counter is incremented.
 
In Fortran, the map directive passes a reference and specifies the extent of the arrays in the declaration and no explicit length information is necessary.

<summary markdown="span"><b>Syntax</b></summary>
    
```fortran
! Explicitly maps data to the GPU.
!$omp target data map(to: a(:), b(:)) map(from: c(:))
    ! Moves this region of code to the GPU.
    !$omp target parallel do
    do i = 1, N 
        c(i) = a(i) + b(i)
    end do
!$omp end target data

```

Note that as with `parallel` and workshare directives, as the target region encompases a single parallel region, the directives are be combined.  However, `target` and `target data map` cannot be combined as the control of the data to/from the GPU and the control of the offloaded code are distinct.

# OpenMP GPU Exercise

Let's continue modifying the code and add the OpenMP offloading directives and constructs.

**Click on the <b>[source code](../source_code/rdf.f90)</b> link, and start modifying the RDF code.**

Remember to introduce both the `target` **and** `target data map` directives to the code in the appropriate places.

Note that without starting from the correct multicore code (check the [solution](../source_code/SOLUTION/rdf_cpu.f90)) and changing it, you will not get the expected outcome after running the below cells. Remember to **SAVE** your code after changes, before running the below cells.

**REMINDER**: To be able to see the Nsight Systems profiler output, please download the latest version of Nsight Systems from [here](https://developer.nvidia.com/nsight-systems).

### Compile and Run for an NVIDIA GPU

Now let us try to recompile the code for NVIDIA GPU and rerun. The only difference in compilation is that now we pass `gpu` value to the `-mp` compiler option.`-mp=gpu`. 

After running the cells, make sure to check the output first. You can inspect part of the compiler feedback for C or Fortran version and see what it's telling us (your compiler feedback will be similar to the one below). 

### Compile the code for GPU

In [ ]:
#compile for GPU
cd ../source_code && printf "Compiling for GPU ...\n" && nvfortran -mp=gpu -Minfo=mp -o rdf_f rdf.f90 -lnvhpcwrapnvtx &&
printf "\nRunning the executable and validating the output\n" && ./rdf_f && cat Pair_entropy.dat

Example compiler feedback:

```
rdf:
    106, Generating map(tofrom:h_g(z_b_12:z_b_13)) 
         Generating map(to:h_x(z_b_0_1:z_b_1),h_y(z_b_4:z_b_5),h_z(z_b_8:z_b_9)) 
pair_gpu:
    176, !$omp target parallel for
        176, Generating "nvkernel_rdf_pair_gpu_F1L176_2" GPU kernel
```

- Line 106 shows maps for the mapped variables are generated `106, Generating map(tofrom:h_g(z_b_12:z_b_13)) Generating map(to:h_x(z_b_0_1:z_b_1),h_y(z_b_4:z_b_5),h_z(z_b_8:z_b_9))`
- Line 176 shows the GPU kernel is generated `176, Generating "nvkernel_rdf_pair_gpu_F1L176_2" GPU kernel`. 

It is very important to inspect the feedback to make sure the compiler is doing what you have asked of it. 

As we had to modifed the code, let's validate the output:

```
s2 value is -2.43191
s2bond value is -3.87015
```

Now Lets profile the code.

In [ ]:
#profile and see output of nvptx
cd ../source_code && nsys profile -t nvtx,cuda --stats=true --force-overwrite true -o rdf_gpu_f ./rdf_f

Download and save the report file by holding down thr Shift key and right-clicking the [report](../source_code/rdf_gpu_f.nsys-rep) link then choosing Save Link As. Once done, open it via the GUI. Have a look at the example expected profiler report below:

**Example screenshot (Fortran code)**

<img src="../../_common/images/f_openmp_gpu.png">


If you expand the CUDA row (_Timeline View_), you can see memory movements as well as kernels. Checkout the NVTX row and compare the execution time for the `Pair_Calculation` for the multicore version and the GPU offload version. In the *example screenshot*, we were able to reduce the timing significantly, however the exact speedups are system dependant so expect some differences to your results.

Feel free to checkout the [solution](../source_code/SOLUTION/rdf_offload.f90) to help you understand better.

# Optional Exercises

## Teams directive
```teams``` directve creates a league of thread teams where the master thread of each team executes the region. Each of these master threads executes sequentially. In other words, teams directive spawn one or more thread teams with the same number of threads. The execution continues on the master threads of each team (redundantly). There is no synchronization allowed between teams. 

OpenMP calls that somewhere a team, which might be a thread on the CPU or maying a CUDA threadblock or OpenCL workgroup. It will choose how many teams to create based on where you're running, only a few on a CPU (like 1 per CPU core) or lots on a GPU (1000's possibly). `teams` allow OpenMP code to scale from small CPUs to large GPUs because each one works completely independently of the other `teams`.

<img src="../../_common/images/openmp_target_teams.png" width="50%" height="50%">

### Distribute
There's a good chance that we don't want the loop to be run redundantly in every master thread of `teams` though, that seems wasteful and potentially dangerous. With the usage of `distribute` construct the iterations of the next loop are broken into groups that are *distributed* to the master threads of the teams. The iterations are distributed statically and there is no guarantee about the order teams will execute. Also it does not generate parallelism/worksharing within the thread teams.

<img src="../../_common/images/openmp_target_distribute.png" width="50%" height="50%">

The example below of simple stencil code shows the usage of `distribute` along with `team`:

<summary markdown="span"><b>Syntax</b></summary>

```fortran
!$omp target teams distribute
do i = 1, N
    do j = 1, N
        ANew(i,j) = A(i) + A(j)
    end do
end do
!$omp end target    
```

<img src="../../_common/images/openmp_teams.png" width="80%" height="80%">

### Work sharing to improve parallelism

As shown in the image, only the master thread performs the computation which is not so optimal in the case of GPU architecture. To solve this problem we will make use of work-sharing as we did before. When any team encounters a worksharing construct, the work inside the construct is divided among the members of the team, and executed cooperatively instead of being executed by every thread. There are many work-sharing constructs defined, one is using `teams distribute` construct: 

<img src="../../_common/images/openmp_teams_for.png" width="80%" height="80%">
    
    
<summary markdown="span"><b>Syntax</b></summary>

```fortran
!$omp target teams distribute parallel do
do i = 1, N
    do j = 1, N
        ANew(i,j) = A(i) + A(j)
    end do
end do
!$omp end target    
```

Lets now add `teams distribute` directives to the `target parallel` construct.

In [ ]:
#compile for GPU
cd ../source_code && printf "Compiling for GPU ...\n" && nvfortran -mp=gpu -Minfo=mp -o rdf_f rdf.f90 -lnvhpcwrapnvtx &&
printf "\nRunning the executable and validating the output\n" && ./rdf_f && cat Pair_entropy.dat

Example compiler feedback using `teams distribute parallel` construct:

```
rdf:
    106, Generating map(tofrom:h_g(z_b_12:z_b_13)) 
         Generating map(to:h_x(z_b_0_1:z_b_1),h_y(z_b_4:z_b_5),h_z(z_b_8:z_b_9)) 
pair_gpu:
    176, !$omp target teams distribute parallel do
        176, Generating "nvkernel_rdf_pair_gpu_F1L176_2" GPU kernel
```

In Fortran the compiler output is almost identical to the previous exercise with a confirmation that the `target teams distribute` construct was used, which can offer finer control.

### Loop construct

In the above examples, the programmer explicitly requests the steps the compiler should take to map parallelism to the target architecture. Another way to expose more parallelism in a program is to allow a compiler to do the mapping onto the target architectures. The HPC compilers' implementation of the loop supports this descriptive model. In the below examples, the programmer specifies the loop regions to be parallelized by the compiler, and the compilers parallelize the loop across teams and threads using the `teams loop` construct, which is a shortcut for specifying a teams construct containing a loop construct and no other statements:

<summary markdown="span"><b>Syntax</b></summary>

```fortran   
!$omp target teams loop
do i = 1, N
    !$omp loop
    do j = 1, N
        ANew(i,j) = A(i) + A(j)
    end do
end do
```

Moreover, further tuning when using a `loop` construct can be done with the `bind` clause, where binding can be one of `teams`, `parallel`, or `thread`. For more information, please visit (OpenMP documentation)[https://www.openmp.org/spec-html/5.1/openmpsu51.html].

Now, lets start modifying the GPU code and replace the `distribute parallel do` with `loop` directives and clauses. Click on the <b>[source](../source_code/rdf.f90)</b> link and start modifying the RDF code. Remember to **SAVE** your code after changes, before running below cells.  
    
After running the cells, make sure to check the output first. You can inspect part of the compiler feedback and see what it's telling us (your compiler feedback will be similar to the below).

**Note** to get similar output to the below cells, it is expected the code before modification is the same as the [solution code](../source_code/SOLUTION/rdf_offload_teams.f90).

### Compile the code for GPU

In [ ]:
#compile for GPU
cd ../source_code && printf "Compiling for GPU ...\n" && nvfortran -mp=gpu -Minfo=mp -o rdf_f rdf.f90 -lnvhpcwrapnvtx &&
printf "\nRunning the executable and validating the output\n" && ./rdf_f && cat Pair_entropy.dat

Example compiler feedback:

```
rdf:
    106, Generating map(tofrom:h_g(z_b_12:z_b_13)) 
         Generating map(to:h_x(z_b_0_1:z_b_1),h_y(z_b_4:z_b_5),h_z(z_b_8:z_b_9)) 
pair_gpu:
    176, !$omp target teams loop
        176, Generating "nvkernel_rdf_pair_gpu_F1L176_2" GPU kernel
             Generating NVIDIA GPU code
          177, Loop parallelized across teams ! blockidx%x
          179, Loop parallelized across threads(128) ! threadidx%x
        176, Generating Multicore code
          177, Loop parallelized across threads
    179, Loop is parallelizable
```

With this construct, we get far more feedback from the compiler on what it is doing.  The outter loop on *line 177* is parallelised over teams `177, Loop parallelized across teams ! blockidx%x`, and the the inner loop  on *line 179* is parallelised over blocks of 128 threads per team `179, Loop parallelized across threads(128) ! threadidx%x`. 

Now, let's profile the code.

In [ ]:
#profile and see output of nvptx
cd ../source_code && nsys profile -t nvtx,cuda --stats=true --force-overwrite true -o rdf_loop2_f ./rdf_f

Download and save the report file by holding down the Shift key and right-clicking the [report](../source_code/rdf_loop_f.nsys-rep) link then choosing Save Link As. Once done, open it via the GUI. Have a look at the example expected profiler report below:

**Example screenshot (loop version)**

<img src="../../_common/images/f_openmp_gpu.png">


If you expand the CUDA row (_Timeline View_), you can see memory movements as well as kernels. Checkout the NVTX row and compare the execution time and workshare for the `Pair_Calculation` between the different clauses and observe how they affected the results.

Feel free to checkout the [solution](../source_code/SOLUTION/rdf_offload_loop.f90) to help you understand better.

## Collapse clause

The collapse clause explicitlty collapses nested loops associated with the loop directive which can help increase parallelism.  The clause must be associated with a loop directive and span two or more nested loops. The number of nested loops to collapse is specified with the parameter `n`.

<summary markdown="span"><b>Syntax</b></summary>
    
```fortran
!$omp parallel do collapse(n)
```


Collapsing the loops means explicitly telling the compiler to parallelise over the combined indecies of both loops.  For example, in the following there are 2 loops we want to collapse with ```collapse(2)```, and, by collapsing them we are explicitly requesting the loop to be treated as a single loop with worksharing over N*N iterations.

<summary markdown="span"><b>Syntax</b></summary>
    
```fortran

!$omp parallel do collapse(2)
do i = 1, N 
    do j = 1, N 
        dx = a(i) + b(j)
    end do
end do
!$omp end parallel

```

Now, lets start further modifying the GPU code and experiment with the `collapse` clause. Click on the <b>[source](../source_code/rdf.cpp)</b> link and start modifying the RDF code. Remember to **SAVE** your code after changes, before running below cells.

After running the cells, make sure to check the output first. You can inspect part of the compiler feedback and see what it's telling us (your compiler feedback will be similar to the below).

**Note** that the `collapse` clause will not show anything different in the compiler output, however it will affect the code. Make sure to inspect the profiler output to better see these differences.  Further, to get similar output to the below cells, it is expected the code before modification is the same as the [solution code](../source/SOLUTIONS/rdf_offload_loop.cpp) from the previous optional exercise.
    

### Compile the code for GPU

In [ ]:
#compile for GPU
cd ../source_code && printf "Compiling for GPU ...\n" && nvfortran -mp=gpu -Minfo=mp -o rdf_f rdf.f90 -lnvhpcwrapnvtx &&
printf "\nRunning the executable and validating the output\n" && ./rdf_f && cat Pair_entropy.dat

Example compiler Feedback (Fortran version):

Inspect the compiler feedback (you should get a similar output as below) and you can see below: 

- *Line 94* shows variables mapped to the device
- *Line 98* shows the GPU kernel is generated

```
rdf:
    106, Generating map(tofrom:h_g(z_b_12:z_b_13)) 
         Generating map(to:h_x(z_b_0_1:z_b_1),h_y(z_b_4:z_b_5),h_z(z_b_8:z_b_9)) 
pair_gpu:
    176, !$omp target teams loop
        176, Generating "nvkernel_rdf_pair_gpu_F1L176_2" GPU kernel
             Generating NVIDIA GPU code
          177, Loop parallelized across teams, threads(128) collapse(2) ! blockidx%x threadidx%x
          178,   ! blockidx%x threadidx%x collapsed
        176, Generating Multicore code
          177, Loop parallelized across threads
```

You can see that on *line 177* and *line 178* the loops are now collapsed `177, Loop parallelized across teams, threads(128) collapse(2) ! blockidx%x threadidx%x` `178,   ! blockidx%x threadidx%x collapsed`. 

It is very important to inspect the feedback to ensure the compiler is doing what you have asked. Now, let's profile the code.

In [ ]:
#profile and see output of nvptx
cd ../source_code && nsys profile -t nvtx,cuda --stats=true --force-overwrite true -o rdf_collapse_f ./rdf_f

Download and save the report file by holding down the Shift key and right-clicking the [report](../source_code/rdf_collapse_f.nsys-rep) link then choosing Save Link As. Once done, open it via the GUI. Have a look at the example expected profiler report below:

**Example screenshot (GPU no collapse)**

<img src="../../_common/images/openmp_loop_fortran.png">

**Example screenshot (GPU collapse)**

<img src="../../_common/images/openmp_collapse_fortran.png">

Have a look at the size of the grid and the Shared Memory executed.  Without collapse the gird size was `<<<6720,1,1>>>` and no shared memory was used.  Using collapse, the grid size is `<<<352800,1,1>>>` (128 times largeer) and the compiler was able to utilise some shared memory, further improving the performance.

Feel free to checkout the solutions for [Fortran](../source_code/SOLUTION/rdf_offload_collapse.f90) version to help you understand better.

# OpenMP Analysis

**Usage Scenarios**
- Legacy codes with a sizeable codebase need to be ported to GPUs with minimal code changes to sequential code. If a compiler doesn’t understand your directives, the code will still compile sequentially. So you have the benefit of maintaining a single code base.
- Developers want to see if the code structure favours  GPU SIMD/SIMT style or as we say test the waters before moving a large piece of code to a GPU.


**Limitations/Constraints**
- Directive-based programming model like OpenMP depends on a compiler to understand and convert your sequential code to CUDA constructs. OpenMP compiler with target offload support is evolving and they it cannot match the best performance that says using CUDA C constructs directly can give. Things like controlling execution at warp level or limiting the register counts etc are some examples.
    
**Which Compilers Support OpenMP on GPU?**
As of March 2020 here are the compilers that support OpenMP on GPU:

| Compiler | Latest Version | Maintained by | Full or Partial Support |
| --- | --- | --- | --- |
| GCC | 12 | Mentor Graphics | 4.5 and 5.0 partial spec supported |
| CCE| latest | Cray | 4.5 partial spec supported | 
| XL | latest | IBM | 4.5 partial spec supported |
| Clang | 13.0 | Community | 4.5 partial spec supported |
| HPC SDK | 22.11 | NVIDIA HPC SDK | 5.0 spec supported |

# Saving the exercise

If you would like to download this lab for later viewing, it is recommended you go to your browser's file menu (not the Jupyter notebook file menu) and save the complete web page.  This will ensure the images are copied down as well. You can also execute the following cell block to create a zip file of the files you have been working on, and download it with the link below.

In [ ]:
cd ..
rm -f _files.zip
zip -r _files.zip *

**After** executing the above zip command, you should be able to download and save the zip file by holding down the Shift key and right-clicking[Here](../_files.zip) then choosing Save Link As.

# Links and Resources
[OpenMP Programming Model](https://computing.llnl.gov/tutorials/openMP/)

[OpenMP Target Directive](https://www.openmp.org/wp-content/uploads/openmp-examples-4.5.0.pdf)

[NVIDIA Nsight System](https://docs.nvidia.com/nsight-systems/)

# Licensing 

Copyright © 2022 OpenACC-Standard.org.  This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.